# Ampere Leak | VoltRelay Energy
Analytical submission covering network performance, service quality, geography, equipment, pricing, and retention. Put this extracted project folder and the original ZIP or eight CSV files in My Drive. Use this notebook with the provided project code.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess,sys,csv
project = Path('/content/drive/MyDrive/ampere-leak-final')
source = Path('/content/drive/MyDrive/ampere-leak-final/dataset')  # or the original ZIP
run = Path('/content/ampere-leak-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
assert source.exists() and (project/'pipeline.py').is_file(), 'Update project and dataset paths first'
subprocess.run([sys.executable,'-m','pip','install','-r',str(project/'requirements.txt')],check=True)

## Data understanding and cleaning
Eight synthetic relational tables cover January 2024 to June 2025. The fact grain is a swap attempt. The pipeline joins required dimensions, corrects documented timestamp shifts, excludes test stations and audits missing measurements. All published tables are aggregate evidence.

In [ ]:
for file,args in [('pipeline.py',[source,run/'evidence']),('report.py',[run/'evidence',run/'deliverables']),('verify.py',[run/'evidence'])]:
    subprocess.run([sys.executable,'-B',str(project/file),*map(str,args)],check=True)
def rows(name):
    with (run/'evidence'/f'{name}.csv').open(encoding='utf-8') as handle:
        return list(csv.DictReader(handle))
from IPython.display import display, Markdown
def preview(name,n=10):
    data=rows(name)[:n]
    if data:
        keys=list(data[0]); lines=['| '+' | '.join(keys)+' |','| '+' | '.join(['---']*len(keys))+' |']
        lines.extend('| '+' | '.join(str(r[k]).replace('|','/').replace('<','&lt;') for k in keys)+' |' for r in data)
        display(Markdown(chr(10).join(lines)))
preview('forensic_audit',30)

## Q1–Q3 · Network performance, service and geography
Use completion counts for financial rates and all attempts for failure rates. Compare all three half-year periods. For station and thermal views review the coverage and denominators alongside the estimated rates.

In [ ]:
import matplotlib.pyplot as plt
monthly=rows('monthly')
fig, axes=plt.subplots(1,3,figsize=(15,4),facecolor='#f3f0e9')
for ax, column, label, scale in zip(axes,['completed','failure_rate','contribution_per_swap'],['Completed swaps','Failure rate (%)','Modeled contribution (₹/swap)'],[1,100,1]):
    ax.plot([r['month'] for r in monthly],[float(r[column])*scale for r in monthly],color='#a45a36',marker='o',markersize=3); ax.set_title(label); ax.set_xticks(range(0,len(monthly),3)); ax.tick_params(axis='x',rotation=45); ax.grid(alpha=.2)
fig.tight_layout(); plt.show()
preview('margin_bridge'); preview('station_priority',8); preview('expansion_waves'); preview('thermal_stratified')

## Q4–Q6 · Equipment, pricing and retention
The pack-range metric requires matching an immediately preceding issue to a same-pack return. Pricing is an observational city comparison. Return models use first-week exposures and a later eligibility window. None of these comparisons alone identifies a causal effect.

In [ ]:
preview('matched_pack_coverage'); preview('supplier_summary'); preview('pricing_did'); preview('fleet_economics',6); preview('retention_logit',7)

## Key findings and action
Unit economics improved year over year, while failures worsened from the preceding half. No charged pack was the largest observed failure category. Test targeted battery float at matched hubs and validate recovery, contribution and rider returns before approving a rollout. Financial scenarios are illustrative and use editable assumptions.

In [ ]:
preview('roi_scenarios',18)
print('Open analysis_report.html and scorecard.html under',run/'deliverables')
from google.colab import files
import shutil
files.download(shutil.make_archive(str(run),'zip',run))